fine-tuning

In [1]:
!pip install -U --quiet peft bitsandbytes transformers accelerate trl PyMuPDF

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
compel 2.3.1 requires transformers~=4.25, but you have transformers 5.3.0 which is incompatible.


In [3]:
from datasets import Dataset, load_dataset

# dataset = load_dataset("roneneldan/TinyStories", split="train")

In [2]:
import fitz

def extract_text_from_pdf(pdf_path):
    text_blocks = []
    with fitz.open(pdf_path) as doc:
        for page in doc:
            text = page.get_text("text").strip()
            if text:
                text_blocks.append(text)
    return text_blocks

In [3]:
text_data = extract_text_from_pdf("LLM Finetuning  02 Introduction of Finetuning.pdf")

In [4]:
text_data

['Process involved in the model Building:\n→\nData Ingestion\n•\nData Analysis\n•\nData Preprocessing\n•\nModel Building\n•\nModel Evaluation\n•\nModel building in Classical Machine Learning: \n→\nSupervised Modelling\n✓\nLinear Regression                \n•\nLogistic Regression\n•\nSupport Vector Machine\n•\nDecision Tree\n•\nRandom Forest\n•\nLight Gradient Boosting Machine(LGBM)\n•\nXtreme Gradient Boosting Machine(XGBM)\n•\nNaïve Bayes\n•\nUnsupervised Modelling\n✓\nClustering: K-Means, Hierarchical Clustering, DB Scan Clustering\n✓\nModel Building in Deep learning\n→\nANN: Artificial Neural Network(ANN)  for Regression and Classification\n✓\nCNN: Convolution Neural Network(CNN) for Grid Like Data Ex. Images\n✓\nWe used CNN-based architectures to solve tasks such as:\nImage Classification\n•\nObject Detection\n•\nObject Segmentation\n•\nObject Tracking\n•\nOptical Character Recognition (OCR)\n•\nImage \nClassification\nLeNet-5, AlexNet, VGG-16/19, ResNet (e.g., ResNet-50), DenseNet

In [5]:
import re
def split_paragraph(pages):
    paragraphs = []
    for page_text in pages:
        # split on double line breaks
        chunks = re.split(r'\n\s\n', page_text)
        for chunk in chunks:
            clean = chunk.strip()
            if len(clean) > 30: # ignore too short lines
                paragraphs.append(clean)
    return paragraphs

In [6]:
chunks = split_paragraph(pages=text_data)

In [7]:
data = [{"text": p} for p in chunks]

In [8]:
data

[{'text': 'Process involved in the model Building:\n→\nData Ingestion\n•\nData Analysis\n•\nData Preprocessing\n•\nModel Building\n•\nModel Evaluation\n•\nModel building in Classical Machine Learning: \n→\nSupervised Modelling\n✓\nLinear Regression                \n•\nLogistic Regression\n•\nSupport Vector Machine\n•\nDecision Tree\n•\nRandom Forest\n•\nLight Gradient Boosting Machine(LGBM)\n•\nXtreme Gradient Boosting Machine(XGBM)\n•\nNaïve Bayes\n•\nUnsupervised Modelling\n✓\nClustering: K-Means, Hierarchical Clustering, DB Scan Clustering\n✓\nModel Building in Deep learning\n→\nANN: Artificial Neural Network(ANN)  for Regression and Classification\n✓\nCNN: Convolution Neural Network(CNN) for Grid Like Data Ex. Images\n✓\nWe used CNN-based architectures to solve tasks such as:\nImage Classification\n•\nObject Detection\n•\nObject Segmentation\n•\nObject Tracking\n•\nOptical Character Recognition (OCR)\n•\nImage \nClassification\nLeNet-5, AlexNet, VGG-16/19, ResNet (e.g., ResNet-50),

In [10]:
from datasets import Dataset

In [11]:
dataset = Dataset.from_list(data)

In [12]:
dataset

Dataset({
    features: ['text'],
    num_rows: 14
})

## Selecting the Model

In [13]:
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T" # for small memory

In [14]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling

In [15]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [16]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [17]:
# preprocess
def tokenize_fn(examples):
    tokens = tokenizer(examples['text'], truncation=True, padding="max_length", max_length=512)
    tokens['labels'] = tokens['input_ids'].copy()
    return tokens

In [18]:
tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=['text'])

Map:   0%|          | 0/14 [00:00<?, ? examples/s]

In [ ]:
tokenized

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 14
})

In [ ]:
model = AutoModelForCausalLM.from_pretrained(model_name)

c:\Users\fajal\anaconda3\envs\traffenv\Lib\site-packages\huggingface_hub\file_download.py:722: UserWarning: Not enough free disk space to download the file. The expected file size is: 4400.22 MB. The target location C:\Users\fajal\.cache\huggingface\hub\models--TinyLlama--TinyLlama-1.1B-intermediate-step-1431k-3T\blobs only has 851.14 MB free disk space.
  warnings.warn(


model.safetensors:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

In [1]:
training_args = TrainingArguments(
    output_dir="./TinyLama_results",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    save_steps=500,
    save_total_limit=2,
    logging_steps=50,
    learning_rate=2e-5,
    fp16=True,
    report_to="none"
)

NameError: name 'TrainingArguments' is not defined

In [ ]:
help(TrainingArguments)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized
)

In [ ]:
trainer.train()

 ---------------------------------------------------------------------------
OutOfMemoryError                          Traceback (most recent call last)
/tmp/ipykernel_4491/4032920361.py in <cell line: 0>()
----> 1 trainer.train()

OutOfMemoryError: CUDA out of memory. Tried to allocate 16.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 1.81 MiB is free. Including non-PyTorch memory, this process has 14.56 GiB memory in use. Of the allocated memory 14.28 GiB is allocated by PyTorch, and 146.96 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

#### We didn't defined what and how much parameters we were finetuning, instead we tried to retrain the whole model

##### We have two methods for partial fine-tuning:
<ol>
    <li>Freeze some layer and finetune unfreezed layer(Old CNN and BERT style)</li>
    <li>LORA(Append some external weight to the already trained pretrained weight)</li>
</ol>

## LORA

In [ ]:
!pip install -U peft bitsandbytes transformers accelerate

In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

In [7]:
import time
for i in range(60):
    print('A minute passed')
    time.sleep(60)

A minute passed
A minute passed
A minute passed
A minute passed
A minute passed
A minute passed
A minute passed
A minute passed


KeyboardInterrupt: 

## Non-Instruction Finetuning Dataset

In [ ]:
dataset = load_dataset("HuggingFaceFW/fineweb")
pubmed = load_dataset("ncbi/pubmed")
dataset = load_dataset("datajuicer/the-pile-pubmed-asbtracts-refined-by-data-juicer")
dataset = load_dataset("open-llm-leaderboard/open_llm_corpus")
owt = load_dataset("Skylion007/openwebtext")
ds = load_dataset("armanc/scientific_papers")

## Instruction Finetuning Dataset

In [ ]:
d1 = "https://huggingface.co/datasets/Amod/mental_health_counseling_conversations"
d2 = "https://huggingface.co/datasets/yahma/alpaca-cleaned"
d3 = "https://huggingface.co/datasets/Open-Orca/OpenOrca"
d4 = "https://huggingface.co/datasets/tatsu-lab/alpaca"
d5 = "https://huggingface.co/datasets/OpenAssistant/oasst1"

## Preference Alignment Dataset

In [ ]:
p1 = "https://huggingface.co/datasets/Anthropic/hh-rlhf"
p2 = "https://huggingface.co/datasets/argilla/ultrafeedback-binarized-preference-cleaned"
p3 = "https://huggingface.co/datasets/xinlai/Math-Step-DPO-10k"